# Train a  DenseNet Model on the SEED Dataset

In this tutorial, we'll demonstrate how you can leverage TorchEEG to train a  DenseNet on the SEED dataset for emotion classification. Let's navigate through this process step-by-step, covering everything from dataset initialization to model evaluation.


## Step 1: Initialize the Dataset
---

#### To begin with, we employ the SEED dataset provided by TorchEEG. Each EEG sample in this dataset lasts for 1 second and contains 200 data points.
#### In terms of offline preprocessing, we perform the following tasks:
- Divide each electrode's EEG signal into 5 frequency sub-bands (Delta, Theta, Alpha, Beta, and Gamma)
- Compute the differential entropy (DE) of each sub-band to serve as features
- Map the extracted features onto a 2D grid based on the SEED channel layout

#### For online processing, we convert the EEG grid representations into PyTorch Tensors, ensuring they are compatible with neural network inputs.
#### Additionally, the emotional labels are extracted and shifted by adding an offset of 1.

In [1]:
from torcheeg.datasets import SEEDDataset
from torcheeg import transforms
from torcheeg.datasets.constants import SEED_CHANNEL_LOCATION_DICT

def increment_label(x):
    return x + 1

dataset = SEEDDataset(
    io_path=f'./examples_seed_dense_net_v2/seed_de_bands', 
    root_path=r'C:\Users\55359\Documents\SEED_EEG\Preprocessed_EEG',
    
    offline_transform=transforms.Compose([
        transforms.BandDifferentialEntropy(
            sampling_rate=200, 
            band_dict={
                "delta": [1, 4],
                "theta": [4, 8],
                "alpha": [8, 14],
                "beta": [14, 31],
                "gamma": [31, 75]
            }
        ),
        
        transforms.ToGrid(SEED_CHANNEL_LOCATION_DICT) 
    ]),
    
    online_transform=transforms.Compose([
        transforms.ToTensor()
    ]),
    
    label_transform=transforms.Compose([
        transforms.Select('emotion'),
        transforms.Lambda(increment_label)
    ]),
    
    chunk_size=200,
    num_worker=4
)

#from torcheeg.datasets import SEEDDataset
#from torcheeg import transforms
#from torcheeg.datasets.constants import SEED_CHANNEL_LOCATION_DICT

#dataset = SEEDDataset(io_path=f'./examples_seed_dense_net_v2/seed',
#                      root_path=r'C:\Users\55359\Documents\SEED_EEG\Preprocessed_EEG',
#                      offline_transform=transforms.Compose([
#                          transforms.BandDifferentialEntropy(sampling_rate=200),
#                          transforms.ToGrid(SEED_CHANNEL_LOCATION_DICT)
#                      ]),
#                      online_transform=transforms.ToTensor(),
#                      label_transform=transforms.Compose([
#                          transforms.Select('emotion'),
#                          transforms.Lambda(lambda x: x + 1)
#                      ]),
#                      chunk_size=200,
#                      num_worker=4)

[2026-07-02 15:07:33] INFO (torcheeg/MainThread) 🔍 | Processing EEG data. Processed EEG data has been cached to ./examples_seed_dense_net_v2/seed_de_bands.
[2026-07-02 15:07:33] INFO (torcheeg/MainThread) ⏳ | Monitoring the detailed processing of a record for debugging. The processing of other records will only be reported in percentage to keep it clean.
[PROCESS]: 100%|██████████| 45/45 [54:52<00:00, 73.17s/it]
[2026-07-02 16:13:25] INFO (torcheeg/MainThread) ✅ | All processed EEG data has been cached to ./examples_seed_dense_net_v2/seed_de_bands.
[2026-07-02 16:13:25] INFO (torcheeg/MainThread) 😊 | Please set io_path to ./examples_seed_dense_net_v2/seed_de_bands for the next run, to directly read from the cache if you wish to skip the data processing step.


## Step 2: Divide the Training and Test Samples in the Dataset
---

#### Next, we partition our dataset into training and test sets using 5-fold cross-validation via `KFoldGroupbyTrial`. We group the data based on their trial index to prevent data leakage, where each trial group contributes 4 folds to the training set and 1 fold to the test set. 
#### These grouped samples are then combined to form the final training and test sets. The split configuration and indices are shuffled with a fixed random seed (42) and cached locally to ensure reproducibility.

In [2]:
from torcheeg.model_selection import KFoldGroupbyTrial
k_fold = KFoldGroupbyTrial(n_splits=5, shuffle=True, random_state=42, split_path=f'./examples_seed_dense_net_v2/split')

## Step 3: Define the Model and Initiate Training
---

#### We loop through each cross-validation fold to initialize the network and execute the training pipeline. In this setup, a standard `DenseNet-121` architecture (configured with a growth rate of 32 and a block configuration of [6, 12, 24, 16]) is employed. 
#### To accommodate the EEG grid structure, the network's input layer (`conv0`) is modified to accept 5-channel features (corresponding to the five frequency bands) with a reduced kernel size of 3 and a stride of 1. Furthermore, the initial max-pooling layer (`pool0`) is replaced with an Identity mapping to preserve spatial resolution for the 9x9 EEG grids.

#### The model is trained for 50 epochs per fold using TorchEEG's `ClassifierTrainer` with a learning rate of 1e-4 and a weight decay of 1e-4. To monitor generalization and prevent overfitting, validation metrics are computed at the end of each epoch. Finally, PyTorch Lightning callbacks are utilized to manage checkpoints, and a multi-fold baseline evaluation is performance-tracked across all partitions.

In [3]:
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from torcheeg.trainers import ClassifierTrainer
from torchvision.models import densenet121
import torch.nn as nn
import logging
import numpy as np
from pytorch_lightning.callbacks import Callback

# 0. Silencing configurations
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("torcheeg").setLevel(logging.ERROR)
pl.seed_everything(42, workers=True)

# Callback to show progress including validation metrics in a single line
class SimpleProgress(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1
        max_epochs = trainer.max_epochs
        
        # Extracting training metrics
        train_loss = trainer.callback_metrics.get('train_loss', 0.0)
        train_acc = trainer.callback_metrics.get('train_accuracy', 0.0)
        
        # Extracting validation metrics (now available because limit_val_batches is enabled)
        val_loss = trainer.callback_metrics.get('val_loss', 0.0)
        val_acc = trainer.callback_metrics.get('val_accuracy', 0.0)
        
        # Updates the line in the same place
        print(f"Epoch {epoch:02d}/{max_epochs} | "
              f"Train Loss: {train_loss:.4f} - Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} - Acc: {val_acc:.4f}", end='\r')

# List to store the Softmax accuracy of each fold
softmax_accuracies = []

for i, (train_dataset, val_dataset) in enumerate(k_fold.split(dataset)):
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    # 1. DenseNet adapted for 9x9 input
    model = densenet121(num_classes=3)
    model.features.conv0 = nn.Conv2d(5, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()

    # 2. Trainer
    trainer = ClassifierTrainer(model=model,
                                num_classes=3,
                                lr=1e-4,
                                weight_decay=1e-4,
                                accelerator="gpu")

    print(f"\n--- Starting Fold {i} | Softmax Baseline ---")

    # 3. Training
    trainer.fit(train_loader,
                val_loader,
                max_epochs=50,
                default_root_dir=f'./examples_seed_dense_net_v2/t/model/{i}',
                callbacks=[
                    pl.callbacks.ModelCheckpoint(save_last=True),
                    SimpleProgress() 
                ],
                enable_progress_bar=False, 
                enable_model_summary=False,
                limit_val_batches=1.0) # CHANGED: 1.0 enables validation so we can trace overfitting

    # 4. Final Fold Evaluation 
    score = trainer.test(val_loader,
                         enable_progress_bar=False,
                         enable_model_summary=False)[0]
    
    fold_softmax_acc = score["test_accuracy"]
    softmax_accuracies.append(fold_softmax_acc)
    
    print(f'\nFold {i} test accuracy (DenseNet121 + Softmax): {fold_softmax_acc:.4f}')

# FINAL SOFTMAX BASELINE RESULTS
print("\n" + "="*50)
print(f"SOFTMAX MULTI-FOLD MEAN ACCURACY: {np.mean(softmax_accuracies):.4f} (+/- {np.std(softmax_accuracies):.4f})")
print("="*50)

Seed set to 42



--- Starting Fold 0 | Softmax Baseline ---


C:\Users\55359\anaconda3\envs\torcheeg\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
C:\Users\55359\anaconda3\envs\torcheeg\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 50/50 | Train Loss: 0.1300 - Acc: 0.9444 | Val Loss: 0.2438 - Acc: 0.9361

C:\Users\55359\anaconda3\envs\torcheeg\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


...

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.9422440528869629
        test_loss           0.21300797164440155
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Fold 3 test accuracy (DenseNet121 + Softmax): 0.9422

--- Starting Fold 4 | Softmax Baseline ---


C:\Users\55359\anaconda3\envs\torcheeg\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
C:\Users\55359\anaconda3\envs\torcheeg\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epoch 50/50 | Train Loss: 0.0252 - Acc: 1.0000 | Val Loss: 0.2412 - Acc: 0.9348

C:\Users\55359\anaconda3\envs\torcheeg\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.9347776174545288
        test_loss           0.24121937155723572
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Fold 4 test accuracy (DenseNet121 + Softmax): 0.9348

SOFTMAX MULTI-FOLD MEAN ACCURACY: 0.9365 (+/- 0.0033)


## Step 4: Hybrid Classification, Training Pipeline, and Evaluation
---

#### We loop through each cross-validation fold, initializing a pre-trained `DenseNet-121` network with a custom 5-channel input layer to process the EEG frequency bands. The model undergoes 50 epochs of fine-tuning via `ClassifierTrainer`. 
#### Immediately following the deep learning training phase, the model's standard Softmax classification performance is evaluated and recorded to serve as a baseline comparison.

#### Next, the network's convolutional blocks are frozen and utilized as a fixed feature extractor. After applying adaptive average pooling, the resulting 1024-dimensional embeddings are extracted, standardized, and fed into a Support Vector Machine (SVM) with an RBF kernel. This hybrid approach allows us to directly compare a deep Softmax baseline against the combination of deep spatial feature learning with the robust classification boundaries of SVMs.

#### Finally, the pipeline evaluates the overall performance. To address the reviewers' constraints, the system automatically exports comprehensive learning curves containing **both training and validation metrics (Loss and Accuracy)** for every fold to verify that overfitting was prevented. It also generates multi-class confusion matrices for the emotional states (Negative, Neutral, and Positive) and outputs a final comparison table displaying the mean accuracy and standard deviation for both the Softmax and SVM methods.

In [4]:
import os
import torch
import torch.nn as nn
import pytorch_lightning as pl
import logging
import warnings # Added to silence hardware warnings
from torch.utils.data import DataLoader
from torchvision.models import densenet121, DenseNet121_Weights
from torcheeg.trainers import ClassifierTrainer
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# SILENCING CONFIGURATIONS (COMPLETE)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("torcheeg").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
logging.getLogger("pytorch_lightning.trainer.connectors.accelerator_connector").setLevel(logging.ERROR)

# Maintains reproducibility
pl.seed_everything(42, workers=True)

# Optimization for RTX 4000
torch.set_float32_matmul_precision('high')

# Directory configuration for saving images
save_path = './examples_seed_dense_net_v2/step4/resultados_seed'
os.makedirs(save_path, exist_ok=True)

# Callback to capture both training and validation logs from each epoch
class TrainingLogger(pl.Callback):
    def __init__(self):
        self.history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        if 'train_loss' in metrics:
            self.history['train_loss'].append(metrics['train_loss'].item())
        if 'train_accuracy' in metrics:
            self.history['train_acc'].append(metrics['train_accuracy'].item())
        if 'val_loss' in metrics:
            self.history['val_loss'].append(metrics['val_loss'].item())
        if 'val_accuracy' in metrics:
            self.history['val_acc'].append(metrics['val_accuracy'].item())

# Function to generate visualizations containing both train and validation curves
def salvar_visualizacoes(history, y_true, y_pred, fold, acc):
    # 1. Validation Curves Plot (Loss and Accuracy)
    plt.figure(figsize=(14, 5))
    
    # Loss subplot (Train vs Val)
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    if history['val_loss']: # Plot validation if available
        plt.plot(history['val_loss'], label='Val Loss', color='red', linestyle='--')
    plt.title(f'Fold {fold} - Learning Loss', fontsize=18)
    plt.xlabel('Epochs', fontsize=18)
    plt.ylabel('Loss', fontsize=18)
    plt.legend(fontsize=14) 
    plt.tick_params(axis='both', labelsize=14)

    # Accuracy subplot (Train vs Val)
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy', color='orange')
    if history['val_acc']: # Plot validation if available
        plt.plot(history['val_acc'], label='Val Accuracy', color='green', linestyle='--')
    plt.title(f'Fold {fold} - Learning Accuracy', fontsize=18)
    plt.xlabel('Epochs', fontsize=18)
    plt.ylabel('Accuracy', fontsize=18)
    plt.legend(fontsize=14) 
    plt.tick_params(axis='both', labelsize=14)
    
    plt.tight_layout()
    plt.savefig(f'{save_path}/treino_fold_{fold}.png')
    plt.close()

    # 2. Confusion Matrix
    plt.figure(figsize=(8, 7)) 
    cm = confusion_matrix(y_true, y_pred)
    
    ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Neg', 'Neu', 'Pos'], 
                yticklabels=['Neg', 'Neu', 'Pos'],
                annot_kws={"size": 18}) 
    
    plt.ylabel('Reality', fontsize=18)
    plt.xlabel('Prediction', fontsize=18)
    ax.tick_params(axis='both', which='major', labelsize=18)
    
    plt.tight_layout()
    plt.savefig(f'{save_path}/matriz_fold_{fold}.png')
    plt.close()


# 3. TRAINING LOOP (COMPARING DENSENET-SOFTMAX VS DENSENET-SVM)
if __name__ == '__main__':
    softmax_accuracies = []
    svm_accuracies = []

    for i, (train_dataset, val_dataset) in enumerate(k_fold.split(dataset)):
        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

        model = densenet121(weights=DenseNet121_Weights.DEFAULT)
        model.features.conv0 = nn.Conv2d(5, 64, kernel_size=3, stride=1, padding=1, bias=False)
        model.features.pool0 = nn.Identity()
        
        num_ftrs = model.classifier.in_features
        model.classifier = nn.Linear(num_ftrs, 3)

        logger_cb = TrainingLogger()

        trainer = ClassifierTrainer(
            model=model,
            num_classes=3,
            lr=1e-4,
            weight_decay=1e-4,
            accelerator="gpu",
            devices=1
        )
        
        print(f"\n--- Starting Fold {i} | Phase 1: Fine-tuning DenseNet (50 Epochs) ---")
        trainer.fit(
            train_loader,
            val_loader,
            max_epochs=50,
            callbacks=[logger_cb],
            enable_progress_bar=False,
            enable_model_summary=False
            # limit_val_batches=0.0 REMOVED: validation is now calculated to generate curves
        )

        # Requisition 1: Extract Softmax Baseline results before moving to SVM
        softmax_score = trainer.test(val_loader, enable_progress_bar=False, enable_model_summary=False)[0]
        softmax_accuracies.append(softmax_score["test_accuracy"])

        print(f"--- Fold {i} | Phase 2: Extracting Features and Training SVM ---")
        model.eval()
        model.cuda()
        
        feature_extractor = nn.Sequential(model.features, nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)))

        def extract(loader):
            x_list, y_list = [], []
            with torch.no_grad():
                for x, y in loader:
                    feat = feature_extractor(x.cuda()).view(x.size(0), -1)
                    x_list.append(feat.cpu().numpy())
                    y_list.append(y.numpy())
            return np.vstack(x_list), np.concatenate(y_list)

        X_train, y_train = extract(train_loader)
        X_val, y_val = extract(val_loader)

        scaler = StandardScaler()
        X_train_std = scaler.fit_transform(X_train)
        X_val_std = scaler.transform(X_val)

        svm = SVC(kernel='rbf', C=1.0, gamma='auto')
        svm.fit(X_train_std, y_train)

        y_pred = svm.predict(X_val_std)
        svm_acc = accuracy_score(y_val, y_pred)
        svm_accuracies.append(svm_acc)
        
        # Generates plots containing BOTH train and validation curves
        salvar_visualizacoes(logger_cb.history, y_val, y_pred, i, svm_acc)
        
        print(f'[PARTIAL RESULT] Fold {i} - Softmax Acc: {softmax_score["test_accuracy"]:.4f} | SVM Acc: {svm_acc:.4f}')

    # FINAL COMPARISON TABLE FOR THE PAPER
    print("\n" + "="*50)
    print(" FINAL COMPARISON TABLE")
    print("="*50)
    print(f"DenseNet121 + Softmax: {np.mean(softmax_accuracies):.4f} (+/- {np.std(softmax_accuracies):.4f})")
    print(f"DenseNet121 + SVM (RBF): {np.mean(svm_accuracies):.4f} (+/- {np.std(svm_accuracies):.4f})")
    print("="*50)
    print(f"Images successfully saved in: {os.path.abspath(save_path)}")
    print("="*50)

Seed set to 42



--- Starting Fold 0 | Phase 1: Fine-tuning DenseNet (50 Epochs) ---
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      test_accuracy         0.9493464231491089
        test_loss           0.18257839977741241
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
--- Fold 0 | Phase 2: Extracting Features and Training SVM ---
[PARTIAL RESULT] Fold 0 - Softmax Acc: 0.9493 | SVM Acc: 0.9517

--- Starting Fold 1 | Phase 1: Fine-tuning DenseNet (50 Epochs) ---
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────